In [1]:
from src.components.data_processing.data_loader import DataLoader, DataUtils
from loguru import logger
import re
import polars.selectors as cs

In [2]:
import os
import s3fs
import polars as pl
os.environ["AWS_ACCESS_KEY_ID"] = 'H9OECGSDIR9AQ9S65ZGV'
os.environ["AWS_SECRET_ACCESS_KEY"] = 'jl6EjjznVonoxzL3eGDhpGMmz1gbkrzN+TXzpnL2'
os.environ["AWS_SESSION_TOKEN"] = 'eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJhY2Nlc3NLZXkiOiJIOU9FQ0dTRElSOUFROVM2NVpHViIsImFsbG93ZWQtb3JpZ2lucyI6WyIqIl0sImF1ZCI6WyJtaW5pby1kYXRhbm9kZSIsIm9ueXhpYSIsImFjY291bnQiXSwiYXV0aF90aW1lIjoxNzc3MDM1NjAyLCJhenAiOiJvbnl4aWEiLCJjbmYiOnsiamt0IjoiSE51MEVHcHVuQ3k1emRjcmhaWExGUWF0TjRPSVNFbWZtZnk1a3k3MlBINCJ9LCJlbWFpbCI6ImFydGh1ci5tYW5jZWF1QHN0dWRlbnQtY3MuZnIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZXhwIjoxNzc3OTA1ODM4LCJmYW1pbHlfbmFtZSI6Ik1hbmNlYXUiLCJnaXZlbl9uYW1lIjoiQXJ0aHVyIiwiZ3JvdXBzIjpbIlVTRVJfT05ZWElBIl0sImlhdCI6MTc3NzMwMTAzOCwiaXNzIjoiaHR0cHM6Ly9hdXRoLmxhYi5zc3BjbG91ZC5mci9hdXRoL3JlYWxtcy9zc3BjbG91ZCIsImp0aSI6Im9ucnRydDo5YWI3N2ZlNC03NDRjLTU0MjAtZjc5Ni0yZDVhYzQ0ZmRiZTciLCJsb2NhbGUiOiJlbiIsIm5hbWUiOiJBcnRodXIgTWFuY2VhdSIsInBvbGljeSI6InN0c29ubHkiLCJwcmVmZXJyZWRfdXNlcm5hbWUiOiJhcnRodXJtYW5jZWF1IiwicmVhbG1fYWNjZXNzIjp7InJvbGVzIjpbIm9mZmxpbmVfYWNjZXNzIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZhdWx0LXJvbGVzLXNzcGNsb3VkIl19LCJyZXNvdXJjZV9hY2Nlc3MiOnsiYWNjb3VudCI6eyJyb2xlcyI6WyJtYW5hZ2UtYWNjb3VudCIsIm1hbmFnZS1hY2NvdW50LWxpbmtzIiwidmlldy1wcm9maWxlIl19fSwicm9sZXMiOlsib2ZmbGluZV9hY2Nlc3MiLCJ1bWFfYXV0aG9yaXphdGlvbiIsImRlZmF1bHQtcm9sZXMtc3NwY2xvdWQiXSwic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBncm91cHMgZW1haWwiLCJzaWQiOiI4OTYwMGU5ZC1jMTg2LTYwY2MtN2Q5Mi05Njg1OTIxZDY2MTIiLCJzdWIiOiJiYzc3Yjk3YS1lN2VkLTQ4ZWMtYmQ4Zi1lNzBjYjZhMTYyZGIiLCJ0eXAiOiJEUG9QIn0.Km_a0cUNOqv8YGCUeMQYknjaxvyV2tf_ruDU5m4QEsDINjFulZThi2suv6OpCutYbIamkHmZyijXuqsZl_X-dQ'
os.environ["AWS_DEFAULT_REGION"] = 'us-east-1'
fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])
storage_options = {
    "aws_access_key_id": os.environ["AWS_ACCESS_KEY_ID"] ,
    "aws_secret_access_key": os.environ["AWS_SECRET_ACCESS_KEY"],
    "aws_region": os.environ["AWS_DEFAULT_REGION"] ,
}

In [3]:
communes = pl.scan_csv('s3://arthurmanceau/election_modeling_uhcp/data/raw/insee_geo/2026/communes_2026.csv', storage_options=storage_options, infer_schema_length=40000).collect()

In [4]:
communes = communes.with_columns(
    DEP_NUM = pl.col('DEP').replace({'2A': 20, '2B': 20}).cast(pl.Int64),
    codecommune = pl.col("COM").cast(pl.String).str.zfill(5)
)

A priori on garde toutes les communes selon toutes les modalités on fera le tri après.

In [5]:
communes

TYPECOM,COM,REG,DEP,CTCD,ARR,TNCC,NCC,NCCENR,LIBELLE,CAN,COMPARENT,DEP_NUM,codecommune
str,str,i64,str,str,str,i64,str,str,str,str,i64,i64,str
"""COM""","""1001""",84,"""1""","""01D""","""12""",5,"""ABERGEMENT CLEMENCIAT""","""Abergement-Clémenciat""","""L'Abergement-Clémenciat""","""108""",null,1,"""01001"""
"""COM""","""1002""",84,"""1""","""01D""","""11""",5,"""ABERGEMENT DE VAREY""","""Abergement-de-Varey""","""L'Abergement-de-Varey""","""101""",null,1,"""01002"""
"""COM""","""1004""",84,"""1""","""01D""","""11""",1,"""AMBERIEU EN BUGEY""","""Ambérieu-en-Bugey""","""Ambérieu-en-Bugey""","""101""",null,1,"""01004"""
"""COM""","""1005""",84,"""1""","""01D""","""12""",1,"""AMBERIEUX EN DOMBES""","""Ambérieux-en-Dombes""","""Ambérieux-en-Dombes""","""122""",null,1,"""01005"""
"""COM""","""1006""",84,"""1""","""01D""","""11""",1,"""AMBLEON""","""Ambléon""","""Ambléon""","""104""",null,1,"""01006"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""COM""","""97613""",6,"""976""","""976R""",null,0,"""M TSANGAMOUJI""","""M'Tsangamouji""","""M'Tsangamouji""","""97613""",null,976,"""97613"""
"""COM""","""97614""",6,"""976""","""976R""",null,1,"""OUANGANI""","""Ouangani""","""Ouangani""","""97610""",null,976,"""97614"""
"""COM""","""97615""",6,"""976""","""976R""",null,0,"""PAMANDZI""","""Pamandzi""","""Pamandzi""","""97611""",null,976,"""97615"""


In [3]:
data_path = "s3://arthurmanceau/election_modeling_uhcp/data/"
elections_to_exclude = []
vote_variables = [
        "ppar",
        "pvoteD",
        "pvoteG",
        "pvoteCG",
        "pvoteCD",
        "pvoteC",
        "pvoteTD",
        "pvoteTG",
        "pvoteGCG",
        "pvoteDCD"
    ] # Extract all a priori

In [27]:
def interpret_election_path(relative_path):
    path_element = relative_path.split(os.sep)[-3:]  # ['legislative', '1848', 'leg1848_csv']
    election_type = path_element[0]  # 'legislative'
    year = path_element[1]  # '1848'
    election_code = path_element[2].split('_')[0] # leg1848
    return election_type, year, election_code

def load_electoral_data():
    """Function that loads all electoral dataset"""
    folder_path = os.path.join(data_path + "raw/", "elections/")
    election_datasets = {}
    xs = (
        DataUtils._create_fs()
        if DataUtils._detect_s3(data_path)
        else os
    )
    for root, dirs, files in xs.walk(folder_path):
        for dir in dirs:
            for root, _, files in xs.walk(os.path.join(folder_path, dir)):
                for file in files:
                    if (
                            file.endswith(".parquet")
                            and (not file.startswith("."))
                            and file not in elections_to_exclude
                    ):
                        election_type, year, election_code = interpret_election_path(os.path.relpath(root, folder_path))
                        logger.debug(f"Processing election : {(election_type, year)}")

                        file_path = DataUtils.path_helper(
                            folder_path, os.path.join(root, file)
                        )
                        election_datasets[election_code] = pl.scan_parquet(file_path).collect()

    return election_datasets

helper_cols = set({'dep','nomdep','codecommune','nomcommune', 'plm'})
def load_socio_economic_data():
    folder_path = os.path.join(data_path, "raw/")
    socio_economic_dataset = {}
    xs = (
        DataUtils._create_fs()
        if DataUtils._detect_s3(data_path)
        else os
    )
    for root, dirs, files in xs.walk(folder_path):
        for dir in dirs:
            if dir != 'elections':
                for root, _, files in xs.walk(os.path.join(folder_path, dir)):
                    for file in files:
                        if (
                                file.endswith(".parquet")
                                and (not file.startswith("."))
                        ):
                            if 'communes' in file:
                                key = 'codecommune'
                            else:
                                key = 'dep'
                            print(key)
                            file_path = DataUtils.path_helper(
                                folder_path, os.path.join(root, file)
                            )
                            data_code = file_path.split('/')[-1].split('.')[0]
                            logger.debug(f"Processing file : {data_code}")
                            
                            df = pl.scan_parquet(file_path).collect()

                            # Identify features
                            all_feature_years = df.select(cs.matches(r".*\d{4}$")).columns

                            # Single unpivot for all features
                            df_long = df.unpivot(
                                on=all_feature_years,
                                index=list(set(df.columns).intersection(helper_cols)),
                                variable_name="variable",
                                value_name="raw",
                            ).with_columns(
                                pl.col("raw").cast(pl.Float64, strict=True),
                                year=pl.col("variable").str.tail(4).cast(pl.Int32),
                                feature_name=pl.col("variable").str.head(-4),
                            ).sort(
                                [key, "feature_name", "year"]
                            ).with_columns(
                                lag=pl.col("raw").shift(1).over(key, "feature_name"),
                                rank=pl.col("raw").rank("dense").over(key, "feature_name"),
                                delta=pl.col("raw").diff(1).over(key, "feature_name"),
                                pct_change=pl.col("raw").pct_change(1).over(key, "feature_name"),
                            )
                            socio_economic_dataset[data_code] = df_long
                            
    return socio_economic_dataset

In [113]:
election_datasets = load_electoral_data()

2026-04-27 15:45:58.649 | DEBUG    | __main__:load_electoral_data:27 - Processing election : ('legislative', '1848')
2026-04-27 15:45:58.839 | DEBUG    | __main__:load_electoral_data:27 - Processing election : ('legislative', '1849')
2026-04-27 15:45:59.050 | DEBUG    | __main__:load_electoral_data:27 - Processing election : ('legislative', '1850')
2026-04-27 15:45:59.271 | DEBUG    | __main__:load_electoral_data:27 - Processing election : ('legislative', '1851')
2026-04-27 15:45:59.463 | DEBUG    | __main__:load_electoral_data:27 - Processing election : ('legislative', '1871')
2026-04-27 15:45:59.627 | DEBUG    | __main__:load_electoral_data:27 - Processing election : ('legislative', '1872')
2026-04-27 15:45:59.799 | DEBUG    | __main__:load_electoral_data:27 - Processing election : ('legislative', '1874')
2026-04-27 15:45:59.961 | DEBUG    | __main__:load_electoral_data:27 - Processing election : ('legislative', '1876')
2026-04-27 15:46:00.164 | DEBUG    | __main__:load_electoral_dat

In [ ]:
socio_economic_data = load_socio_economic_data()

2026-04-27 16:38:27.785 | DEBUG    | __main__:load_socio_economic_data:63 - Processing file : agesexcommunes


codecommune


In [145]:
socio_economic_data['menagescommunes']

dep,nomdep,codecommune,nomcommune,plm,nmen,nmencomp,pmencomp,permencomp
str,str,str,str,i64,i64,i64,f64,f64
"""01""","""AIN""","""01001""","""ABERGEMENT-CLEMENCIAT""",0,124,3,0.024194,0.001428
"""01""","""AIN""","""01002""","""ABERGEMENT-DE-VAREY""",0,53,3,0.056604,0.279386
"""01""","""AIN""","""01004""","""AMBERIEU-EN-BUGEY""",0,3200,159,0.049688,0.165093
"""01""","""AIN""","""01005""","""AMBERIEUX-EN-DOMBES""",0,243,12,0.049383,0.15957
"""01""","""AIN""","""01006""","""AMBLEON""",0,29,5,0.1724138,0.957667
…,…,…,…,…,…,…,…,…
"""95""","""VAL-D'OISE""","""95676""","""VILLERS-EN-ARTHIES""",0,97,5,0.051546,0.190179
"""95""","""VAL-D'OISE""","""95678""","""VILLIERS-ADAM""",0,206,15,0.072816,0.565188
"""95""","""VAL-D'OISE""","""95680""","""VILLIERS-LE-BEL""",0,6710,540,0.080477,0.664478


In [139]:
socio_economic_data['agesexcommunes']

dep,nomdep,codecommune,nomcommune,poph0141962,poph15391962,poph40591962,poph60p1962,popf0141962,popf15391962,popf40591962,popf60p1962,ageh1962,agef1962,poph0141968,popf0141968,poph15391968,popf15391968,poph40591968,popf40591968,poph60p1968,popf60p1968,ageh1968,agef1968,poph0141975,popf0141975,poph15391975,popf15391975,poph40591975,popf40591975,poph60p1975,popf60p1975,ageh1975,agef1975,poph0141982,popf0141982,poph15391982,…,perprop60p1950,perpropf1951,perage1951,perprop0141951,perprop60p1951,perpropf1952,perage1952,perprop0141952,perprop60p1952,perpropf1953,perage1953,perprop0141953,perprop60p1953,perpropf1954,perage1954,perprop0141954,perprop60p1954,perpropf1955,perage1955,perprop0141955,perprop60p1955,perpropf1956,perage1956,perprop0141956,perprop60p1956,perpropf1957,perage1957,perprop0141957,perprop60p1957,perpropf1958,perage1958,perprop0141958,perprop60p1958,perpropf1959,perage1959,perprop0141959,perprop60p1959
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""25""","""DOUBS""","""25443""","""PALANTINE""",5.0,5.0,6.0,3.0,7.0,6.0,5.0,6.0,35.157894,34.916668,0.0,0.0,0.0,0.0,4.0,4.0,8.0,8.0,67.0,65.333336,5.0,0.0,0.0,5.0,5.0,10.0,10.0,5.0,44.5,53.25,12.0,0.0,4.0,…,0.09306,0.946069,0.006167,0.929899,0.09306,0.946069,0.006167,0.929899,0.09306,0.946069,0.006167,0.929899,0.09306,0.946069,0.006167,0.929899,0.09306,0.946069,0.006167,0.929899,0.09306,0.946069,0.006167,0.929899,0.09306,0.946069,0.006167,0.929899,0.09306,0.946069,0.006167,0.929899,0.09306,0.946069,0.006167,0.929899,0.09306
"""70""","""HAUTE-SAONE""","""70440""","""RECOLOGNE""",3.0,6.0,4.0,2.0,3.0,5.0,4.0,5.0,32.666668,40.529411,0.0,0.0,0.0,4.0,8.0,4.0,0.0,8.0,52.0,55.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,0.0,4.0,…,0.711124,0.112636,0.088098,0.355285,0.711124,0.112636,0.088098,0.355285,0.711124,0.112636,0.088098,0.355285,0.711124,0.112636,0.088098,0.355285,0.711124,0.112636,0.088098,0.355285,0.711124,0.112636,0.088098,0.355285,0.711124,0.112636,0.088098,0.355285,0.711124,0.112636,0.088098,0.355285,0.711124,0.112636,0.088098,0.355285,0.711124
"""21""","""COTE-D'OR""","""21276""","""FONTAINES-EN-DUESMOIS""",32.0,37.0,24.0,16.0,29.0,22.0,21.0,17.0,32.229359,33.797752,20.0,20.0,36.0,16.0,24.0,16.0,24.0,20.0,35.846153,37.555557,10.0,30.0,20.0,15.0,15.0,5.0,10.0,25.0,37.454544,38.0,0.0,4.0,4.0,…,0.18882,0.043178,0.144777,0.93831,0.18882,0.043178,0.144777,0.93831,0.18882,0.043178,0.144777,0.93831,0.18882,0.043178,0.144777,0.93831,0.18882,0.043178,0.144777,0.93831,0.18882,0.043178,0.144777,0.93831,0.18882,0.043178,0.144777,0.93831,0.18882,0.043178,0.144777,0.93831,0.18882,0.043178,0.144777,0.93831,0.18882
"""31""","""HAUTE-GARONNE""","""31266""","""LAHAGE""",6.0,94.0,17.0,10.0,8.0,14.0,10.0,11.0,28.574802,41.534885,12.0,0.0,120.0,8.0,4.0,16.0,20.0,4.0,27.641026,49.857143,10.0,15.0,30.0,20.0,10.0,5.0,10.0,10.0,35.333332,33.0,12.0,16.0,64.0,…,0.089764,0.001752,0.149059,0.003273,0.089764,0.001752,0.149059,0.003273,0.089764,0.001752,0.149059,0.003273,0.089764,0.001752,0.149059,0.003273,0.089764,0.001752,0.149059,0.003273,0.089764,0.001752,0.149059,0.003273,0.089764,0.001752,0.149059,0.003273,0.089764,0.001752,0.149059,0.003273,0.089764,0.001752,0.149059,0.003273,0.089764
"""06""","""ALPES-MARITIMES""","""06076""","""LIEUCHE""",1.0,3.0,4.0,2.0,2.0,5.0,4.0,1.0,40.5,35.75,0.0,0.0,4.0,4.0,0.0,4.0,0.0,0.0,27.0,44.5,0.0,0.0,0.0,0.0,10.0,0.0,0.0,5.0,49.5,62.0,4.0,4.0,8.0,…,0.323348,0.509821,0.861277,0.069635,0.323348,0.509821,0.861277,0.069635,0.323348,0.509821,0.861277,0.069635,0.323348,0.509821,0.861277,0.069635,0.323348,0.509821,0.861277,0.069635,0.323348,0.509821,0.861277,0.069635,0.323348,0.509821,0.861277,0.069635,0.323348,0.509821,0.861277,0.069635,0.323348,0.509821,0.861277,0.069635,0.323348
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

In [138]:
unique_cols = set([col[:-4] for col in socio_economic_data['agesexcommunes'].columns])
print(unique_cols)

{'', 'perpropf', 'poph1539', 'poph014', 'popf60p', 'nomcom', 'no', 'prop60p', 'poph4059', 'poph60p', 'propf1539', 'codecom', 'poph', 'perprop014', 'prop1539', 'popf014', 'perage', 'ageh', 'prop014', 'propf60p', 'agef', 'propf014', 'prop4059', 'perprop60p', 'propf', 'popf', 'popf4059', 'popf1539', 'propf4059', 'age', 'pop'}


In [137]:
socio_economic_data['agesexcommunes'].select(cs.starts_with("poph014"))

poph0141962,poph0141968,poph0141975,poph0141982,poph0141990,poph0141999,poph0142006,poph0142011,poph0142016,poph0141960,poph0141961,poph0141963,poph0141964,poph0141965,poph0141966,poph0141967,poph0141969,poph0141970,poph0141971,poph0141972,poph0141973,poph0141974,poph0141976,poph0141977,poph0141978,poph0141979,poph0141980,poph0141981,poph0141983,poph0141984,poph0141985,poph0141986,poph0141987,poph0141988,poph0141989,poph0141991,poph0141992,poph0141993,poph0141994,poph0141995,poph0141996,poph0141997,poph0141998,poph0142000,poph0142001,poph0142002,poph0142003,poph0142004,poph0142005,poph0142007,poph0142008,poph0142009,poph0142010,poph0142012,poph0142013,poph0142014,poph0142015,poph0142017,poph0142018,poph0142019,poph0142020,poph0142021,poph0142022
f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64
5.0,0.0,5.0,12.0,0.0,0.0,4.0,4.0,20.0,7,6,4.0,3.0,3.0,2.0,1.0,1.0,1.0,2.0,3.0,4.0,4.0,6.0,7.0,8.0,9.0,10.0,11.0,11.0,9.0,8.0,6.0,5.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,3.0,3.0,4.0,4.0,4.0,4.0,7.0,10.0,14.0,17.0,23,26,30,33,36,39
3.0,0.0,0.0,0.0,0.0,0.0,0.0,5.090909,0.0,4,4,3.0,2.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,4.0,4.0,3.0,2.0,1.0,0,0,0,0,0,0
32.0,20.0,10.0,0.0,12.0,8.0,20.267523,7.875,0.0,36,34,30.0,28.0,26.0,24.0,22.0,19.0,17.0,16.0,14.0,13.0,11.0,9.0,7.0,6.0,4.0,3.0,1.0,2.0,3.0,5.0,6.0,8.0,9.0,11.0,12.0,11.0,11.0,10.0,10.0,9.0,9.0,8.0,10.0,12.0,13.0,15.0,17.0,19.0,18.0,15.0,13.0,10.0,6.0,5.0,3.0,2.0,0,0,0,0,0,0
6.0,12.0,10.0,12.0,24.0,8.0,15.304348,8.2089615,5.4664903,4,5,7.0,8.0,9.0,10.0,11.0,12.0,11.0,11.0,11.0,11.0,10.0,10.0,11.0,11.0,11.0,11.0,12.0,14.0,15.0,17.0,18.0,20.0,21.0,23.0,22.0,20.0,19.0,17.0,15.0,13.0,12.0,10.0,9.0,10.0,11.0,12.0,13.0,14.0,14.0,12.0,11.0,10.0,8.0,7.0,7.0,6.0,5,4,4,3,3,2
1.0,0.0,0.0,4.0,0.0,4.0,0.0,4.0,0.0,1,1,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,3.0,3.0,4.0,3.0,3.0,2.0,2.0,1.0,1.0,0.0,1.0,1.0,2.0,2.0,3.0,3.0,4.0,3.0,3.0,2.0,2.0,1.0,1.0,1.0,2.0,2.0,3.0,3.0,2.0,2.0,1.0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
85.0,92.0,75.0,84.0,60.0,60.0,63.84119,52.603092,null,83,84,86.0,87.0,89.0,90.0,91.0,90.0,87.0,85.0,82.0,80.0,77.0,76.0,78.0,79.0,80.0,81.0,83.0,81.0,78.0,75.0,72.0,69.0,66.0,63.0,60.0,60.0,60.0,60.0,60.0,60.0,60.0,60.0,61.0,61.0,62.0,62.0,63.0,63.0,62.0,59.0,57.0,55.0,null,null,null,null,0,0,0,0,0,0
8.0,8.0,null,null,null,null,null,null,null,8,8,8.0,8.0,8.0,8.0,8.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0
84.0,108.0,55.0,72.0,112.0,84.0,67.913139,95.999779,null,76,80,88.0,92.0,96.0,100.0,104.0,100.0,93.0,85.0,78.0,70.0,63.0,57.0,60.0,62.0,65.0,67.0,70.0,77.0,82.0,87.0,92.0,97.0,102.0,107.0,109.0,106.0,103.0,100.0,96.0,93.0,90.0,87.0,82.0,79.0,77.0,75.0,73.0,70.0,74.0,79.0,85.0,90.0,null,null,null,null,0,0,0,0,0,0


In [135]:
out = socio_economic_data['agesexcommunes'].unpivot(
    on=[
        "poph0141962", "poph15391962", "poph40591962", "poph60p1962",
        "popf0141962", "popf15391962", "popf40591962", "popf60p1962",
        "ageh1962", "agef1962",
        "poph0141968", "popf0141968",
        # ... add all time-varying columns here
    ],
    index=["dep", "nomdep", "codecommune", "nomcommune"],
    variable_name="variable",
    value_name="value",
)
print(out)

shape: (455_820, 6)
┌─────┬─────────────────┬─────────────┬────────────────────────┬─────────────┬───────┐
│ dep ┆ nomdep          ┆ codecommune ┆ nomcommune             ┆ variable    ┆ value │
│ --- ┆ ---             ┆ ---         ┆ ---                    ┆ ---         ┆ ---   │
│ str ┆ str             ┆ str         ┆ str                    ┆ str         ┆ f64   │
╞═════╪═════════════════╪═════════════╪════════════════════════╪═════════════╪═══════╡
│ 25  ┆ DOUBS           ┆ 25443       ┆ PALANTINE              ┆ poph0141962 ┆ 5.0   │
│ 70  ┆ HAUTE-SAONE     ┆ 70440       ┆ RECOLOGNE              ┆ poph0141962 ┆ 3.0   │
│ 21  ┆ COTE-D'OR       ┆ 21276       ┆ FONTAINES-EN-DUESMOIS  ┆ poph0141962 ┆ 32.0  │
│ 31  ┆ HAUTE-GARONNE   ┆ 31266       ┆ LAHAGE                 ┆ poph0141962 ┆ 6.0   │
│ 06  ┆ ALPES-MARITIMES ┆ 06076       ┆ LIEUCHE                ┆ poph0141962 ┆ 1.0   │
│ …   ┆ …               ┆ …           ┆ …                      ┆ …           ┆ …     │
│ 23  ┆ CREUSE         

In [134]:
socio_economic_data['agesexcommunes'].schema

Schema([('dep', String),
        ('nomdep', String),
        ('codecommune', String),
        ('nomcommune', String),
        ('poph0141962', Float64),
        ('poph15391962', Float64),
        ('poph40591962', Float64),
        ('poph60p1962', Float64),
        ('popf0141962', Float64),
        ('popf15391962', Float64),
        ('popf40591962', Float64),
        ('popf60p1962', Float64),
        ('ageh1962', Float64),
        ('agef1962', Float64),
        ('poph0141968', Float64),
        ('popf0141968', Float64),
        ('poph15391968', Float64),
        ('popf15391968', Float64),
        ('poph40591968', Float64),
        ('popf40591968', Float64),
        ('poph60p1968', Float64),
        ('popf60p1968', Float64),
        ('ageh1968', Float64),
        ('agef1968', Float64),
        ('poph0141975', Float64),
        ('popf0141975', Float64),
        ('poph15391975', Float64),
        ('popf15391975', Float64),
        ('poph40591975', Float64),
        ('popf40591975', Float64)